# Test Smell - Multi‑Agent Notebook

This Colab notebook performs **test smell detection and refactoring** using a **multi‑agent workflow**.

**How it works**

* **Dependencies & runtime** – The first two code cells install all Python libraries and start an Ollama server directly in Colab (or connect to an existing one via `base_url`).
* **Parameter widgets** – Fill in the CSV containing the test cases, the path to the smell‑definition file, choose the smell to analyse, the Ollama `model` (e.g. `phi4`, `llama3`), sampling `temperature`, server `base_url`, and `max_iters`.
* **Main loop** – For each test case, the notebook runs a detection–refinement workflow involving up to four agents. The first agent identifies whether the smell is present, the second reviews that decision, and if confirmed, the third agent proposes a refactoring. The fourth agent then evaluates whether the refactored code is correct and smell-free. This cycle can repeat up to `max_iters` times (3). All outcomes are appended to the original CSV to preserve traceability and allow inspection of intermediate steps.



In [ ]:
# @title Install dependencies
%pip uninstall -q -y langchain langchain-core langchain-community langchain-ollama langchain-text-splitters
%pip install -q --no-deps lightrag[ollama]
%pip install -q "langchain==0.3.*" "langchain-core==0.3.*" "langchain-community==0.3.*" "langchain-ollama==0.3.*"

In [ ]:
# @title Install and start Ollama runtime
!sudo apt-get -qq update && sudo apt-get -y -qq install pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

import os, threading, subprocess, time
def start_ollama():
    os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
    os.environ["OLLAMA_ORIGINS"] = "*"
    subprocess.Popen(["ollama", "serve"])
threading.Thread(target=start_ollama, daemon=True).start()
time.sleep(5)
print("=== verificação ===")
!ollama --version

In [ ]:
# @title Create parameter widgets
import ipywidgets as widgets
from IPython.display import display

csv_path_widget = widgets.Text(value="Dataset.csv", description='CSV:')
definitions_path_widget = widgets.Text(value="test_smell_definitions_and_refactorings.txt", description='Defs:')
smell = "Magic Number Test" # @param ["Assertion Roulette", "Magic Number Test", "Duplicate Assert", "Exception Handling", "Conditional Test Logic"]
smell_widget = widgets.Text(value=smell, description='Smell:')
model_widget = widgets.Text(value="gemma4:31b", description='Model:')
temperature_widget = widgets.FloatSlider(value=0.6, min=0.0, max=1.0, step=0.1, description='Temp:')
base_url_widget = widgets.Text(value="http://localhost:11434", description='Base URL:')
max_iters_widget = widgets.IntText(value=3, description='Max Iters:')

display(csv_path_widget, definitions_path_widget, model_widget,
        temperature_widget, base_url_widget, max_iters_widget)

Text(value='Dataset.csv', description='CSV:')

Text(value='test_smell_definitions_and_refactorings.txt', description='Defs:')

Text(value='gemma4:31b', description='Model:')

FloatSlider(value=0.6, description='Temp:', max=1.0)

Text(value='http://localhost:11434', description='Base URL:')

IntText(value=3, description='Max Iters:')

In [ ]:
# @title Capture parameters from widgets
csv_path = csv_path_widget.value
definitions_path = definitions_path_widget.value
smell = smell_widget.value
model = model_widget.value
temperature = temperature_widget.value
base_url = base_url_widget.value
max_iters = max_iters_widget.value

In [ ]:
# @title Define prompt templates and helpers
from __future__ import annotations

import argparse
import os
import re
import shutil
import subprocess
import sys
import threading
import time
from datetime import date
from pathlib import Path

import pandas as pd
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain



def load_smell(def_text: str, smell_name: str) -> tuple[str, str]:
    pat = re.compile(
        r'test_smell_name\s*=\s*"([^"]+)"\s*[\r\n]+'
        r'test_smell_definition\s*=\s*"""(.*?)"""',
        re.S,
    )
    for name, definition in pat.findall(def_text):
        if name.strip().lower() == smell_name.lower():
            return name.strip(), definition.strip()
    raise ValueError(f'Didnt find Smell "{smell_name}".')

In [ ]:
!pip install tqdm
from tqdm.notebook import tqdm

In [ ]:
# @title Run detection and evaluation loop

code = "{code}"
agent_feedback = "{agent_feedback}"
agent_answer = "{agent_answer}"
refactored_code = "{refactored_code}"
reviewer_feedback = "{reviewer_feedback}"

def build_prompts(test_smell_name: str, test_smell_definition_and_refactoring: str) -> dict[str, PromptTemplate]:
    detect = PromptTemplate(
        input_variables=["code", "agent_feedback"],
        template=f"""
You are a coding assistant with many years of experience that detects test smells.
{test_smell_definition_and_refactoring}

Your goal is to determine if the provided test code exhibits the test smell "{test_smell_name}".
{code}
Next I may give you further details.
{agent_feedback}
If the test code contains {test_smell_name}, respond with EXACTLY "YES" on the first line and explain why. Ignore code comments. If it does not contain, say EXACTLY "NO" on the first line and explain why not.
"""
    )
    eval_detect = PromptTemplate(
        input_variables=["code", "agent_answer"],
        template=f"""
You are a coding expert reviewing the detection of a test smell. Consider the following test smell.
{test_smell_definition_and_refactoring}

A previous agent analyzed the following test code.
{code}
It gave the following answer.
{agent_answer}
Your goal is to evaluate if the previous detection by another agent is correct and justified. Ignore code comments.
If you do not agree, answer NO and explain what's wrong with it and what to correct.
If yes, just say YES.
"""
    )
    refactor = PromptTemplate(
        input_variables=["code", "reviewer_feedback"],
        template=f"""
You are a coding assistant specializing in test code analysis and refactoring, with many years of experience.

{test_smell_definition_and_refactoring}

Your task is as follows.
First analyze the provided test code to resolve test smell occurrences "{test_smell_name}". If there is no smell, output the original code unchanged.
Second ensure the test preserves the same behavior, but is free of {test_smell_name}.
Third output only the final refactored code, valid under JUnit 5.
Finally check the refactored version does not introduce compilation errors.

Provide only the final refactored code, with no additional explanation or text.
Code to analyze:
{code}

Next I may provide you further details.
{reviewer_feedback}
"""
    )
    eval_refactor = PromptTemplate(
        input_variables=["refactored_code"],
        template=f"""
You are a code reviewer specializing in JUnit 5 test smells.
{test_smell_definition_and_refactoring}

Analyze the following code.
{refactored_code}

Your task is to check three conditions.
First check the code does not have the test smell {test_smell_name}.
Second verify the code follows JUnit 5 specification.
Finally confirms the code does not have compilation errors.

If the code satisfy all conditons, respond with EXACTLY "YES" on the first line.
If not, respond with EXACTLY "NO" on the first line, then explain in one or two sentences why.

Let's think step by step.
"""
    )
    return {
        "detect": detect,
        "eval_detect": eval_detect,
        "refactor": refactor,
        "eval_refactor": eval_refactor,
    }


with open(definitions_path, "r", encoding="utf-8") as f:
    definitions_text = f.read()

smell_name, smell_def = load_smell(definitions_text, smell)


!ollama pull {model}
prompt = build_prompts(smell_name, smell_def)
llm = OllamaLLM(model=model, base_url=base_url, temperature=temperature)
chains = {
    k: LLMChain(llm=llm, prompt=v) for k, v in prompt.items()
}

df = pd.read_csv(csv_path)
today_str = date.today().strftime("%Y-%m-%d")
for c in ["LLM", "Test Smell", "Date"]:
    if c not in df.columns:
        df[c] = ""

with open("agentes.txt", "w", encoding="utf-8") as log:
  original_stdout = sys.stdout
  sys.stdout = log
  try:
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing tests"):
        code = "@Test\n" + str(row.get("Test Code", "")).strip()
        df.loc[idx, ["LLM", "Test Smell", "Date"]] = [model, smell_name, today_str]

        explanation = ""
        confirmed = False

        print(f"\n===== Test {idx+1} =====\n")
        print(f"Test Code: \n {code}")

        for it in range(1, max_iters + 1):
            r1 = chains["detect"].run(code=code, agent_feedback=explanation)
            r2 = chains["eval_detect"].run(code=code, agent_answer=r1)

            df.at[idx, f"{it}° - Detected smell?"] = "YES" if "yes" in r1.lower() else "NO"
            df.at[idx, f"{it}° - Do you agree with detection?"] = "YES" if "yes" in r2.lower() else "NO"

            print(f"[Detection Iter {it}]")
            print(f"1.{it}. Agent 1 Result: {r1}\n===========\n")
            print(f"1.{it}. Agent 2 Result: {r2}\n===========\n")
            print("------------------------------------------------")

            if "yes" in r2.lower():
                confirmed, explanation = True, r1
                break
            explanation = r1
        if not confirmed:
            print("Test smell not confirmed. Skipping refactoring.")
            continue
        if not "yes" in r1.lower():
            print("Test smell not detected. Skipping refactoring.")
            continue

        current, feedback = code, explanation
        for it in range(1, max_iters + 1):
            ref = chains["refactor"].run(code=current, reviewer_feedback=feedback)
            chk = chains["eval_refactor"].run(refactored_code=ref)
            df.at[idx, f"{it}° - Refactored code"] = ref
            ok = "yes" in chk.lower()
            df.at[idx, f"{it}° -  LLM - Is there still a test smell?"] = "NO" if ok else "YES"

            print(f"2.{it}. Agent 3 Result: {ref}\n===========\n")
            print(f"2.{it}. Agent 4 Result: {chk}\n===========\n")
            print("=================================================")

            if ok:
                break
            feedback = "\n".join(chk.splitlines()[1:]) or "No explanation"
            current = ref

  finally:
    sys.stdout = original_stdout

output_csv = f"/content/output_{smell.replace(' ', '_')}.csv"
df.to_csv(output_csv, index=False)
print(f"Results saved to: {output_csv}")
print("Agent log written to agentes.txt")

In [ ]:
from google.colab import files

log_renomeado = f"agentes_multi_{smell.replace(' ', '_')}.txt"
!cp agentes.txt {log_renomeado}

files.download(output_csv)
files.download(log_renomeado)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>